In [2]:
import torch
from transformers import AutoProcessor
from multilingualmc.translator.transformers_customized.models.seamless_m4t.modeling_seamless_m4t import SeamlessM4TModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = AutoProcessor.from_pretrained("facebook/hf-seamless-m4t-Large", cache_dir="/data/user_data/jiaruil5/.cache/", use_fast=False)
model = SeamlessM4TModel.from_pretrained("facebook/hf-seamless-m4t-Large", cache_dir="/data/user_data/jiaruil5/.cache/").to(device)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/data/user_data/jiaruil5/miniconda3/envs/mmc/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
src = '- Compared to highly-optimized tiny CNN object detectors, YOLOS achieves competitive performance in terms of AP, FLOPs and FPS. It could serve as a promising starting point for Transformer-based model scaling in object detection. Handling of Variable Input Sizes: \n - Unlike image classification, object detection benchmarks usually have variable image resolutions and aspect ratios.'
tgt = '- 与高度优化的小型CNN目标检测器相比，YOLOS在AP、FLOPs和FPS方面表现出竞争力。它可以作为基于Transformer的目标检测模型扩展的有希望的起点。处理可变输入大小：\n- 与图像分类不同，目标检测基准通常具有可变的图像分辨率和纵横比。'

In [26]:
import jieba
import itertools
sent_src, sent_tgt = src.split(), [i for i in jieba.cut(tgt, cut_all=False)]
token_src, token_tgt = [processor.tokenizer.tokenize(word) for word in sent_src], [processor.tokenizer.tokenize(word) for word in sent_tgt]
wid_src, wid_tgt = [processor.tokenizer.convert_tokens_to_ids(x) for x in token_src], [processor.tokenizer.convert_tokens_to_ids(x) for x in token_tgt]
ids_src, ids_tgt = processor.tokenizer.prepare_for_model(list(itertools.chain(*wid_src)), return_tensors='pt', model_max_length=processor.tokenizer.model_max_length, truncation=True)['input_ids'].to(device), processor.tokenizer.prepare_for_model(list(itertools.chain(*wid_tgt)), return_tensors='pt', truncation=True, model_max_length=processor.tokenizer.model_max_length)['input_ids'].to(device)
sub2word_map_src = []
for i, word_list in enumerate(token_src):
  sub2word_map_src += [i for x in word_list]
sub2word_map_tgt = []
for i, word_list in enumerate(token_tgt):
  sub2word_map_tgt += [i for x in word_list]


# alignment
align_layer = 21
threshold = 1e-3
model.eval()
with torch.no_grad():
  out_src = model.text_encoder(ids_src.unsqueeze(0), output_hidden_states=True).hidden_states[align_layer][0, 1:-1]
  out_tgt = model.text_encoder(ids_tgt.unsqueeze(0), output_hidden_states=True).hidden_states[align_layer][0, 1:-1]

  dot_prod = torch.matmul(out_src, out_tgt.transpose(-1, -2))

  softmax_srctgt = torch.nn.Softmax(dim=-1)(dot_prod)
  softmax_tgtsrc = torch.nn.Softmax(dim=-2)(dot_prod)

  softmax_inter = (softmax_srctgt > threshold)*(softmax_tgtsrc > threshold)

align_subwords = torch.nonzero(softmax_inter, as_tuple=False)
align_words = set()
for i, j in align_subwords:
  align_words.add( (sub2word_map_src[i], sub2word_map_tgt[j]) )

# printing
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[96m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'

for i, j in sorted(align_words):
  print(f'{color.BOLD}{color.BLUE}{sent_src[i]}{color.END}==={color.BOLD}{color.RED}{sent_tgt[j]}{color.END}')

Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens': False} not recognized.
Keyword arguments {'add_special_tokens':

-===-
Compared===与
Compared===，
highly-optimized===优化
CNN===小型
object===目标
detectors,===检测器
YOLOS===YOLOS
achieves===在
competitive===作为
performance===竞争力
in===出
AP,===、
FLOPs===FLOPs
and===和
FPS.===FPS
FPS.===方面
could===可以
as===基于
starting===的
point===起点
for===。
Transformer-based===Transformer
Transformer-based===的
model===Transformer
scaling===扩展
in===的
detection.===检测
Handling===可变
Input===输入
Sizes:===输入
-===-
Unlike===与
classification,===分类
object===目标
detection===检测
benchmarks===基准
have===具有
image===的
resolutions===分辨率
ratios.===和


In [21]:
len(model.text_encoder(ids_src.unsqueeze(0), output_hidden_states=True).hidden_states)

25

In [20]:
ids_src, ids_tgt

(tensor([     3, 256026,    135, 208137,     61,    243, 175997, 247711, 127765,
          38334,  98516,  74287, 109586, 202836,  14531, 247681,    380,  22272,
           8568,   5055,     78,   6087,  17478,  48399,  61190,     70, 105194,
            290,  26978, 247681,  86742,  34712, 247669,    447,    211,  12503,
         247676,   3080,  27246,  38349,    443,     10,   6765,  21173, 182757,
          20950,    334,  15327,  43463, 247711, 100303,  12654,   3654,  20697,
             70, 109586,    733,  42538, 247676,  13765,   5320,    290,  95630,
           4094,    640,   7251,  17155,     18, 247813,    135,    681,  11352,
          90237,  44004,  42999, 247681, 109586,    733,  42538, 204193, 183340,
         112383,   4942, 127601,  90237,  43498,   6499,    447, 100718,  54044,
             44, 247676,      3], device='cuda:0'),
 tensor([     3, 256026,    135,  74144,  38561, 249126, 247661, 250708, 249320,
          10091,  25909, 249893,  74287, 247661,  35788, 